In [3]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jaccard

In [4]:
df_meta = pd.read_csv('metadata.txt', sep = '\t', usecols = ['MIT_Accession','Metagenomic_file_name'])
df_meta = df_meta.dropna()
df_meta['Metagenomic_file_name'] = df_meta['Metagenomic_file_name'].str.replace('X', '')
df_meta['Metagenomic_file_name'] = df_meta['Metagenomic_file_name'].str.replace('.', '-')
df_meta = df_meta.rename(columns={'Metagenomic_file_name': 'file'})
df_meta

,MIT_Accession,file
2,23-1064,250728Pat_D25-10256
3,23-1065,250728Pat_D25-10257
4,23-1066,250728Pat_D25-10260
5,23-1067,250728Pat_D25-10261
6,23-1069,250513Pat_D25-7850
...,...,...
79,24-0091,250728Pat_D25-10255
80,24-0092,250728Pat_D25-10258
81,24-0093,250728Pat_D25-10259
86,24-0098,250930Pat_D25-12477


In [5]:
df_mgx = pd.read_csv('nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt', sep = '\t', usecols = ['file','taxid_rep'])
df_mgx = df_mgx.drop_duplicates()
df_mgx

,file,taxid_rep
0,250513Pat_D25-7849,37923
1,250513Pat_D25-7849,1261
3,250513Pat_D25-7849,1747
4,250513Pat_D25-7849,423477
6,250513Pat_D25-7849,40324
...,...,...
318505,250930Pat_D25-12478,511
318529,250930Pat_D25-12478,129817
318542,250930Pat_D25-12478,74829
318577,250930Pat_D25-12478,2078786


In [6]:
dfm = pd.merge(df_mgx,df_meta, on = 'file', how = 'left')
dfm = dfm.drop('file', axis=1)
dfm = dfm.rename(columns={'taxid_rep': 'taxid'})
dfm['method'] = 'mgx'
dfm

,taxid,MIT_Accession,method
0,37923,23-1068,mgx
1,1261,23-1068,mgx
2,1747,23-1068,mgx
3,423477,23-1068,mgx
4,40324,23-1068,mgx
...,...,...,...
8935,511,24-0116,mgx
8936,129817,24-0116,mgx
8937,74829,24-0116,mgx
8938,2078786,24-0116,mgx


In [7]:
df_culture = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', usecols = ['MIT_Accession', 'best_taxid'])
df_culture = df_culture.rename(columns={'best_taxid': 'taxid'})
df_culture['method'] = 'culture'
df_culture

,MIT_Accession,taxid,method
0,23-1062,84163,culture
1,23-1062,1382,culture
2,23-1062,1624,culture
3,23-1062,2047,culture
4,23-1062,43675,culture
...,...,...,...
1014,24-0133,1302,culture
1015,24-0133,28037,culture
1016,24-0133,1234680,culture
1017,24-0133,1304,culture


In [8]:
dfm = pd.concat([dfm, df_culture], ignore_index=True)
dfm

,taxid,MIT_Accession,method
0,37923,23-1068,mgx
1,1261,23-1068,mgx
2,1747,23-1068,mgx
3,423477,23-1068,mgx
4,40324,23-1068,mgx
...,...,...,...
9954,1302,24-0133,culture
9955,28037,24-0133,culture
9956,1234680,24-0133,culture
9957,1304,24-0133,culture


In [16]:
# Create binary table
binary_table = pd.crosstab(
    index=[dfm['MIT_Accession'], dfm['method']],
    columns=dfm['taxid']
)

# Convert counts >0 to 1
binary_table = (binary_table > 0).astype(int)

# Reset index to make 'file' and 'method' normal columns
binary_table = binary_table.reset_index()

taxid MIT_Accession   method  24  165  197  199  210  239  253  285  ...  \
0           23-1062  culture   0    0    0    0    0    0    0    0  ...   
1           23-1063  culture   0    0    0    0    0    0    0    0  ...   
2           23-1064  culture   0    0    0    0    0    0    0    0  ...   
3           23-1064      mgx   0    0    0    0    0    0    0    0  ...   
4           23-1065  culture   0    0    0    0    0    0    0    0  ...   
..              ...      ...  ..  ...  ...  ...  ...  ...  ...  ...  ...   
162         24-0131      mgx   0    0    1    0    1    0    0    0  ...   
163         24-0132  culture   0    0    0    0    1    0    0    0  ...   
164         24-0132      mgx   0    0    0    0    0    0    0    0  ...   
165         24-0133  culture   0    0    0    0    0    0    0    0  ...   
166         24-0133      mgx   0    0    0    0    0    0    0    0  ...   

taxid  3415987  3416180  3418415  3418416  3418421  3418555  3421959  3422304  \
0     

In [26]:
# Feature columns
feature_cols = binary_table.columns.difference(['taxid', 'MIT_Accession', 'method'])

# MIT_Accession with both methods
valid_accessions = binary_table.groupby('MIT_Accession')['method'].nunique()
valid_accessions = valid_accessions[valid_accessions == 2].index

# Filter dataframe
df_filtered = binary_table[binary_table['MIT_Accession'].isin(valid_accessions)]

results = []

for acc in valid_accessions:
    acc_data = df_filtered[df_filtered['MIT_Accession'] == acc]
    culture_row = acc_data[acc_data['method'] == 'culture'][feature_cols].values[0]
    mgx_row = acc_data[acc_data['method'] == 'mgx'][feature_cols].values[0]

    # Skip if both rows are all zeros
    if culture_row.sum() == 0 and mgx_row.sum() == 0:
        continue

    # Jaccard similarity
    jac_sim = 1 - jaccard(culture_row, mgx_row)

    # Shared and unique features
    shared_features = ((culture_row != 0) & (mgx_row != 0)).sum()
    unique_culture = ((culture_row != 0) & (mgx_row == 0)).sum()
    unique_mgx = ((mgx_row != 0) & (culture_row == 0)).sum()

    # Total non-zero features across both methods
    total_nonzero_features = ((culture_row != 0) | (mgx_row != 0)).sum()
    
    # Total features in each method
    total_culture_features = (culture_row != 0).sum()
    total_mgx_features = (mgx_row != 0).sum()

    results.append({
        'MIT_Accession': acc,
        'jaccard_similarity': jac_sim,
        'total_nonzero_features': total_nonzero_features,
        'shared_features': shared_features,
        'unique_culture': unique_culture,
        'unique_mgx': unique_mgx,
        'total_culture_features': total_culture_features,
        'total_mgx_features': total_mgx_features
    })

# Convert to DataFrame
jaccard_df = pd.DataFrame(results)

jaccard_df.to_csv('culture_vs_mgx_stats_species.txt', sep = '\t', index=False)